# Dataloader batches

Two shuffled batches (batch size 4, fixed seed) from every dataloader in `dataloaders/`. Images are contrast-stretched per channel (1-99.5 percentile); 2-channel images are shown as a green/magenta composite. Instance masks are overlaid in colour in a second row, track points are drawn as crosses.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

ROOTS = {
    "c2c12": Path("D:/datasets/c2c12"),
    "cellx_predict": REPO / "data/cellx_predict",
    "qpi_cell_death": Path("D:/datasets/qpi_cell_death"),
    "hemato_fate": Path("D:/datasets/hemato_fate"),
    "alfi": Path("D:/datasets/alfi"),
    "celldetective": Path("D:/datasets/celldetective"),
    "xenopus": Path("D:/datasets/xenopus"),
    "p53_fate": Path("D:/datasets/p53_fate"),
}
BATCH_SIZE, N_BATCHES, SEED = 4, 2, 0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader


def batches(ds, collate_fn=None):
    """The first N_BATCHES shuffled batches of ds (num_workers=0: works in notebooks on Windows)."""
    g = torch.Generator().manual_seed(SEED)
    loader = DataLoader(ds, BATCH_SIZE, shuffle=True, generator=g, num_workers=0, collate_fn=collate_fn)
    return [b for b, _ in zip(loader, range(N_BATCHES), strict=False)]


def to_rgb(image):
    """[C, H, W] tensor -> [H, W, 3] in [0, 1], stretched per channel; 2 channels -> green/magenta."""
    x = image.float().numpy()
    lo, hi = np.nanpercentile(x, (1, 99.5), axis=(1, 2), keepdims=True)
    x = np.clip((x - lo) / np.maximum(hi - lo, 1e-6), 0, 1)
    if len(x) == 1:
        return np.repeat(x[0, ..., None], 3, -1)
    return np.stack([x[1], x[0], x[1]], -1)  # ch 0 green, ch 1 magenta


def show_batch(batch, titles, masks=None, points=None, step=1, suptitle=""):
    """One column per sample: image (+ points), and a second row with the instance mask overlaid.

    masks: [B, H, W] instance labels (0 = background); points: per sample [N, 2] (x, y), NaN allowed;
    step: show every step-th pixel (downscaling for large frames).
    """
    images = batch["image"]
    n, rows = len(images), 1 if masks is None else 2
    fig, axes = plt.subplots(rows, n, figsize=(3.2 * n, 3.0 * rows), squeeze=False, layout="constrained")
    for i in range(n):
        rgb = to_rgb(images[i])[::step, ::step]
        axes[0, i].imshow(rgb)
        axes[0, i].set_title(titles[i], fontsize=8)
        if points is not None:
            p = np.asarray(points[i], np.float32).reshape(-1, 2) / step
            axes[0, i].scatter(
                p[:, 0], p[:, 1], c=np.arange(len(p)) % 10, cmap="tab10", vmin=0, vmax=9, s=30, marker="x", linewidths=1
            )
        if masks is not None:
            m = masks[i].numpy()[::step, ::step]
            axes[1, i].imshow(rgb)
            axes[1, i].imshow(
                np.ma.masked_where(m == 0, m % 20),
                cmap="tab20",
                vmin=0,
                vmax=19,
                alpha=0.5,
                interpolation="nearest",
            )
            axes[1, i].set_title(f"{len(np.unique(m[m > 0]))} instances", fontsize=8)
    for ax in axes.flat:
        ax.set_axis_off()
    fig.suptitle(f"{suptitle}  image {tuple(images.shape)}  range [{images.min():.2f}, {images.max():.2f}]", fontsize=9)
    plt.show()

## C2C12

Phase contrast, 1040 x 1392 (shown at half size). Target: disk instance masks around the computer-tracked centroids; crosses are the centroids (`points`). `c2c12_collate` keeps the per-frame point lists as lists.

In [ ]:
from dataloaders.c2c12_dataset import C2C12Dataset, c2c12_collate

ds = C2C12Dataset(ROOTS["c2c12"], frame_step=10)
print(f"{len(ds)} frames from {ds.sequences}")
for k, b in enumerate(batches(ds, c2c12_collate)):
    titles = [f"{f} frame {t}, {len(p)} cells" for f, t, p in zip(b["field"], b["frame"], b["points"], strict=True)]
    show_batch(b, titles, masks=b["target"], points=b["points"], step=2, suptitle=f"C2C12 batch {k}")

## cellX-predict

64 x 64 nucleus crops, GFP (green) / RFP (magenta), no masks. `patches` are unlabelled; `glimpses` are frames of time-lapses ending in apoptosis or mitosis.

In [ ]:
from dataloaders.cellx_predict_dataset import EVENTS, CellXPredictDataset

ds = CellXPredictDataset(ROOTS["cellx_predict"], subset="patches")
for k, b in enumerate(batches(ds)):
    show_batch(
        b,
        [f"max GFP {x[0].max():.2f}, RFP {x[1].max():.2f}" for x in b["image"]],
        suptitle=f"cellX patches ({len(ds)}) batch {k}",
    )

ds = CellXPredictDataset(ROOTS["cellx_predict"], subset="glimpses")
for k, b in enumerate(batches(ds)):
    titles = [
        f"{EVENTS[e]} {g.split('_RFP')[0]}\nframe {f} (t={t}, cutoff {c})"
        for e, g, f, t, c in zip(b["event"], b["glimpse"], b["frame"], b["t"], b["cutoff"], strict=True)
    ]
    show_batch(b, titles, suptitle=f"cellX glimpses ({len(ds)} frames) batch {k}")

## QPI cell death

Quantitative phase (pg/um^2), 600 x 600. Target: sparse instance masks of the annotated cells; the subtitle counts death types (from `ds.labels`) of the cells in the mask.

In [ ]:
from dataloaders.qpi_cell_death_dataset import DEATH_TYPES, QPICellDeathDataset

ds = QPICellDeathDataset(ROOTS["qpi_cell_death"], frame_step=10)
print(f"{len(ds)} frames from {len(ds.sequences)} time-lapses")
for k, b in enumerate(batches(ds)):
    titles = []
    for line, tr, fov, t, m in zip(b["cell_line"], b["treatment"], b["fov"], b["frame"], b["target"], strict=True):
        labels = ds.labels[ds.sequences.index((line, tr, int(fov)))]
        types = [DEATH_TYPES[labels[int(c)][0]].split("/")[0] for c in m.unique() if c > 0 and int(c) in labels]
        counts = ", ".join(f"{types.count(d)} {d}" for d in sorted(set(types)))
        titles.append(f"{line} {tr} fov {int(fov)} frame {int(t)}\n{counts or 'no annotated cells'}")
    show_batch(b, titles, masks=b["target"], suptitle=f"QPI batch {k}")

## HematoFatePrediction

Bright-field, 1040 x 1388 (shown at half size), flat-field corrected. No masks; crosses are the 2 manually tracked cells (`ds.cell_ids`, blue = first, orange = second), absent where not tracked in that frame.

In [ ]:
from dataloaders.hemato_fate_dataset import HematoFateDataset

ds = HematoFateDataset(ROOTS["hemato_fate"], flatfield=True)
print(f"{len(ds)} frames, tracked cells {ds.cell_ids}")
for k, b in enumerate(batches(ds)):
    titles = [
        f"pos {int(p)} t {int(t)}, {int((~pt.isnan().any(1)).sum())} tracked"
        for p, t, pt in zip(b["position"], b["timepoint"], b["points"], strict=True)
    ]
    show_batch(b, titles, points=b["points"], step=2, suptitle=f"Hemato batch {k}")

## ALFI

DIC, 1024 x 1280 (shown at half size). `annotation="dtl"` (MI sequences): every cell boxed as Interphase / Mitosis, instance target = connected components of the semantic masks (nuclei; whole cells in mitosis). `annotation="pheno"` (all 29 sequences): boxes only around cells in EarlyMitosis / LateMitosis / CellDeath / Multipolar; box corners are drawn as crosses.

In [ ]:
from dataloaders.alfi_dataset import CLASSES, ALFIDataset, alfi_collate

for annotation in ("dtl", "pheno"):
    ds = ALFIDataset(ROOTS["alfi"], annotation=annotation)
    print(f"{annotation}: {len(ds)} frames from {len(ds.sequences)} sequences")
    for k, b in enumerate(batches(ds, alfi_collate)):
        titles = [
            f"{s} frame {f}\n" + (", ".join(CLASSES[annotation][i] for i in sorted(set(lab.tolist()))) or "no boxes")
            for s, f, lab in zip(b["sequence"], b["frame"], b["labels"], strict=True)
        ]
        corners = [bx.reshape(-1, 2) for bx in b["boxes"]]  # (xmin, ymin), (xmax, ymax) per box
        show_batch(b, titles, masks=b.get("target"), points=corners, step=2, suptitle=f"ALFI {annotation} batch {k}")

## Celldetective

Instance-segmentation sets of the ADCC co-culture (MCF-7 nuclei, primary NK cells; 4-5 fluorescence / bright-field channels) and the RICM spreading assay (adhesion, bright-field). Sizes vary, so each sample is randomly cropped to 480 x 480. Shown are the 2 channels that best show the annotated population (first green, second magenta).

In [ ]:
from dataloaders.c2c12_dataset import RandomCropFlip
from dataloaders.celldetective_dataset import CelldetectiveDataset

SHOW = {
    "db_mcf7_nuclei_w_lymphocytes": ("live_nuclei_channel", "dead_nuclei_channel"),
    "db_primary_NK_w_mcf7": ("effector_fluo_channel", "live_nuclei_channel"),
    "db_spreading_lymphocytes": ("adhesion_channel", "brightfield_channel"),
}
for subset, channels in SHOW.items():
    crop = RandomCropFlip(480, generator=torch.Generator().manual_seed(SEED))
    ds = CelldetectiveDataset(ROOTS["celldetective"], subset, channels=channels, transform=crop)
    for k, b in enumerate(batches(ds)):
        titles = [f"{n[:30]}\n{c:.4f} um/px" for n, c in zip(b["name"], b["spatial_calibration"], strict=True)]
        show_batch(b, titles, masks=b["target"], suptitle=f"{subset} ({len(ds)} images) batch {k}")

## Xenopus mucociliary epithelium

Membrane (green) / nuclei (magenta) z-slices of the 3 segmented stacks (Figure 2 of S-BSST2777), 1956 x 1956 (shown at quarter size). Target: the slice of the authors' 3D membrane (cells) or nucleus segmentation.

In [ ]:
from dataloaders.xenopus_dataset import XenopusDataset

for target in ("membrane", "nuclei"):
    ds = XenopusDataset(ROOTS["xenopus"], target=target)
    for k, b in enumerate(batches(ds)):
        titles = [f"{tp} z {int(z)}" for tp, z in zip(b["timepoint"], b["z"], strict=True)]
        show_batch(b, titles, masks=b["target"], step=4, suptitle=f"Xenopus {target} batch {k}")

## p53 cell fate

Greyscale frames of the authors' tracking videos (1166 x 1602 for MNNG1, shown at half size; every 10th frame, 10 min/frame); the coloured tracking overlay (crosses, lineage numbers) is burned in and stays visible in grey. Only MNNG1 has a matching lineage table (all 8 Dryad tables are the same file): crosses are its track points. p53_RNAi is shown as an example of the unlabelled conditions.

In [ ]:
from dataloaders.p53_fate_dataset import P53FateDataset, p53_collate

for condition in ("MNNG1", "p53_RNAi"):
    ds = P53FateDataset(ROOTS["p53_fate"], conditions=[condition], frame_step=10)
    for k, b in enumerate(batches(ds, p53_collate)):
        points = b.get("points")
        titles = [f"frame {f} ({(f - 1) * 10} min)" for f in b["frame"]]
        if points is not None:
            titles = [f"{t}, {len(p)} tracked" for t, p in zip(titles, points, strict=True)]
        show_batch(b, titles, points=points, step=2, suptitle=f"p53 {condition} ({len(ds)} frames) batch {k}")